# Export Graph to Parquet
Run this notebook once after rebuilding the graph in Neo4j.
It exports nodes and edges to parquet files that the Streamlit app loads at runtime.

In [ ]:
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    'bolt://127.0.0.1:7687',
    auth=('neo4j', '205spotify')
)

def run_query(query, **kwargs):
    with driver.session(database='neo4j') as session:
        result = session.run(query, **kwargs)
        return pd.DataFrame([r.values() for r in result], columns=result.keys())

print('Connected')

## Export Nodes

In [ ]:
nodes = run_query("""
    MATCH (s:Song)
    RETURN
        s.id            AS track_id,
        s.track_name    AS track_name,
        s.artists       AS artists,
        s.genre         AS genre,
        s.popularity    AS popularity,
        s.community     AS community,
        s.betweenness   AS betweenness,
        s.danceability  AS danceability,
        s.energy        AS energy,
        s.loudness      AS loudness,
        s.speechiness   AS speechiness,
        s.acousticness  AS acousticness,
        s.instrumentalness AS instrumentalness,
        s.liveness      AS liveness,
        s.valence       AS valence,
        s.tempo         AS tempo,
        s.mode          AS mode
""")

print(f'Nodes: {len(nodes)}')
nodes.head(3)

In [ ]:
nodes.to_parquet('data/nodes.parquet', index=False)
print('Saved data/nodes.parquet')

## Export Edges

In [ ]:
# Export in batches to avoid memory issues
import math

batch_size = 100000
total_edges = run_query('MATCH ()-[e:SIMILAR_TO]->() RETURN count(e) AS n').iloc[0]['n']
print(f'Total edges: {total_edges}')

batches = []
for skip in range(0, total_edges, batch_size):
    batch = run_query(f"""
        MATCH (a:Song)-[e:SIMILAR_TO]->(b:Song)
        RETURN a.id AS source, b.id AS target,
               e.similarity AS similarity, e.cost AS cost
        SKIP {skip} LIMIT {batch_size}
    """)
    batches.append(batch)
    print(f'  Fetched {min(skip + batch_size, total_edges)}/{total_edges}')

edges = pd.concat(batches, ignore_index=True)
print(f'Edges loaded: {len(edges)}')
edges.head(3)

In [ ]:
edges.to_parquet('data/edges.parquet', index=False)
print('Saved data/edges.parquet')

## Verify

In [ ]:
n = pd.read_parquet('data/nodes.parquet')
e = pd.read_parquet('data/edges.parquet')
print(f'Nodes: {len(n)}, Edges: {len(e)}')
print(f'Communities: {n["community"].nunique()}')
print(f'Songs with betweenness: {n["betweenness"].notna().sum()}')
driver.close()